# Chương 4 - Phần 1: Khởi Tạo Mã Băm Wavelet Hashing

**Thông tin học viên:**
- **Họ và tên:** Nguyễn Ngọc Anh  
- **Lớp:** CN22H  
- **Môn học:** Thị giác Máy tính (Computer Vision)  

---

## 1. Import Thư Viện & Cài Đặt Ban Đầu

Import các thư viện và cấu hình cơ bản.

In [ ]:
import os
import json
import cv2
import numpy as np
import pywt
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 11
print("Đã import thành công các thư viện!")
print("Phiên bản PyWavelets:", pywt.__version__)
print("Phiên bản OpenCV:", cv2.__version__)

## 2. Chuẩn Bị Dữ Liệu Kiểm Thử

Chúng ta đọc 4 ảnh gốc và sinh ra các biến thể tương tự của `anh.jpg` (xoay ảnh, đổi độ sáng, thêm nhiễu) để tạo tập dữ liệu so khớp.

In [ ]:
import shutil
img_dir = "../data/input"
os.makedirs(img_dir, exist_ok=True)

ch3_demo_path = "../../chapter03_edge_detection/data/input/demo.jpg"
target_demo_path = os.path.join(img_dir, "demo.jpg")
if os.path.exists(ch3_demo_path) and not os.path.exists(target_demo_path):
    shutil.copy(ch3_demo_path, target_demo_path)

base_images = ["anh.jpg", "anh1.jpg", "anh2.jpg", "anh3.jpg", "demo.jpg", "anh4.jpg", "anh5.jpg", "anh6.jpg", "anh7.jpg", "anh8.jpg", "anh9.jpg", "anh10.jpg", "anh11.jpg", "anh12.jpg", "anh13.jpg"]

var_dir = os.path.join(img_dir, "variants")
os.makedirs(var_dir, exist_ok=True)

all_images = {}
image_paths = {}
metadata = []

for idx, base_name in enumerate(base_images):
    base_path = os.path.join(img_dir, base_name)
    if not os.path.exists(base_path):
        continue
    
    img_bgr = cv2.imread(base_path)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = img_gray.shape
    
    # Save original
    name = f"base_{idx}_var_0"
    out_path = os.path.join(var_dir, f"{name}.jpg")
    cv2.imwrite(out_path, img_gray)
    all_images[name] = img_gray
    image_paths[name] = out_path
    metadata.append({"name": name, "group": idx})
    
    # 1. Rotated 15
    img_rotated_15 = cv2.warpAffine(img_gray, cv2.getRotationMatrix2D((w/2, h/2), 15, 1.0), (w, h))
    name = f"base_{idx}_var_1"
    out_path = os.path.join(var_dir, f"{name}.jpg")
    cv2.imwrite(out_path, img_rotated_15)
    all_images[name] = img_rotated_15
    image_paths[name] = out_path
    metadata.append({"name": name, "group": idx})
    
    # 2. Rotated 30
    img_rotated_30 = cv2.warpAffine(img_gray, cv2.getRotationMatrix2D((w/2, h/2), 30, 1.0), (w, h))
    name = f"base_{idx}_var_2"
    out_path = os.path.join(var_dir, f"{name}.jpg")
    cv2.imwrite(out_path, img_rotated_30)
    all_images[name] = img_rotated_30
    image_paths[name] = out_path
    metadata.append({"name": name, "group": idx})
    
    # 3. Bright +40
    img_bright_up = np.clip(img_gray.astype(np.int16) + 40, 0, 255).astype(np.uint8)
    name = f"base_{idx}_var_3"
    out_path = os.path.join(var_dir, f"{name}.jpg")
    cv2.imwrite(out_path, img_bright_up)
    all_images[name] = img_bright_up
    image_paths[name] = out_path
    metadata.append({"name": name, "group": idx})
    
    # 4. Bright -40
    img_bright_down = np.clip(img_gray.astype(np.int16) - 40, 0, 255).astype(np.uint8)
    name = f"base_{idx}_var_4"
    out_path = os.path.join(var_dir, f"{name}.jpg")
    cv2.imwrite(out_path, img_bright_down)
    all_images[name] = img_bright_down
    image_paths[name] = out_path
    metadata.append({"name": name, "group": idx})
    
    # 5. Noise 15
    noise_15 = np.random.normal(0, 15, img_gray.shape).astype(np.int16)
    img_noisy_15 = np.clip(img_gray.astype(np.int16) + noise_15, 0, 255).astype(np.uint8)
    name = f"base_{idx}_var_5"
    out_path = os.path.join(var_dir, f"{name}.jpg")
    cv2.imwrite(out_path, img_noisy_15)
    all_images[name] = img_noisy_15
    image_paths[name] = out_path
    metadata.append({"name": name, "group": idx})
    
    # 6. Noise 40
    noise_40 = np.random.normal(0, 40, img_gray.shape).astype(np.int16)
    img_noisy_40 = np.clip(img_gray.astype(np.int16) + noise_40, 0, 255).astype(np.uint8)
    name = f"base_{idx}_var_6"
    out_path = os.path.join(var_dir, f"{name}.jpg")
    cv2.imwrite(out_path, img_noisy_40)
    all_images[name] = img_noisy_40
    image_paths[name] = out_path
    metadata.append({"name": name, "group": idx})
    
    # 7. Blurred
    img_blurred = cv2.GaussianBlur(img_gray, (11, 11), 0)
    name = f"base_{idx}_var_7"
    out_path = os.path.join(var_dir, f"{name}.jpg")
    cv2.imwrite(out_path, img_blurred)
    all_images[name] = img_blurred
    image_paths[name] = out_path
    metadata.append({"name": name, "group": idx})

print(f"Đã sinh các biến thể ảnh tương tự thành công! Tổng số ảnh: {len(image_paths)}")

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
axes = axes.ravel()
keys_to_plot = list(all_images.keys())[:20]
for i, name in enumerate(keys_to_plot):
    axes[i].imshow(all_images[name], cmap='gray')
    axes[i].set_title(name, fontsize=9)
    axes[i].axis('off')
for j in range(len(keys_to_plot), len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

## 3. Định Nghĩa Thuật Toán Băm Wavelet Hashing

Chúng ta cài đặt cả phương pháp băm thô (Slide) và phương pháp băm nhận thức chuẩn (Standard wHash).

In [ ]:
def custom_quantize(c, step=2):
    if isinstance(c, tuple):
        return tuple(custom_quantize(sub_c, step) for sub_c in c)
    elif isinstance(c, np.ndarray):
        return np.round(c / step).astype(np.int64)
    else:
        return int(np.round(c / step))

def flatten_coeffs(coeffs):
    flat = []
    for c in coeffs:
        if isinstance(c, tuple):
            for sub_c in c: flat.append(sub_c.flatten())
        else:
            flat.append(c.flatten())
    return np.concatenate(flat)

def wavelet_hash_slide(image_path, wavelet='db4', level=1, step=2, size=(64, 64)):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img_resized = cv2.resize(img, size)
    coeffs = pywt.wavedec2(img_resized.astype(np.float32), wavelet, level=level)
    coeffs_quant = [custom_quantize(c, step) for c in coeffs]
    flattened = flatten_coeffs(coeffs_quant)
    return [int(bit % 2) for bit in flattened]

def wavelet_hash_standard(image_path, wavelet='db4', level=1, size=(64, 64)):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img_resized = cv2.resize(img, size)
    coeffs = pywt.wavedec2(img_resized.astype(np.float32), wavelet, level=level)
    cA = coeffs[0]
    med = np.median(cA)
    return (cA > med).astype(int).flatten().tolist()

In [ ]:
def hamming_distance(hash1, hash2):
    # Tối ưu hóa bằng NumPy (vectorization) thay thế vòng lặp Python
    return int(np.sum(np.abs(np.array(hash1) - np.array(hash2))))

image_keys = list(image_paths.keys())

hashes_slide = {name: wavelet_hash_slide(path) for name, path in image_paths.items()}
hashes_std = {name: wavelet_hash_standard(path) for name, path in image_paths.items()}

pairs_results = []
print(f"Đang thực hiện so khớp chéo {len(image_keys) * (len(image_keys) - 1) // 2} cặp ảnh...")

for i in range(len(image_keys)):
    for j in range(i + 1, len(image_keys)):
        name1, name2 = image_keys[i], image_keys[j]
        
        # Cùng nhóm gốc (cùng idx của base_images) -> Tương tự
        group1 = metadata[i]["group"]
        group2 = metadata[j]["group"]
        is_similar = 1 if group1 == group2 else 0
        
        dist_slide = hamming_distance(hashes_slide[name1], hashes_slide[name2])
        dist_std = hamming_distance(hashes_std[name1], hashes_std[name2])
        
        pairs_results.append({
            "pair": [name1, name2],
            "gt": is_similar,
            "dist_slide": dist_slide,
            "dist_std": dist_std
        })

print(f"Đã tính toán xong khoảng cách Hamming cho {len(pairs_results)} cặp ảnh.")

# Hiển thị 15 cặp ảnh đầu tiên làm ví dụ mẫu
print(f"\n{'-'*95}")
print(f"{'Cặp Ảnh':<45} | {'Ground Truth':<12} | {'Khoảng Cách Slide':<15} | {'Khoảng Cách wHash Chuẩn':<15}")
print(f"{'-'*95}")
for item in pairs_results[:15]:
    n1, n2 = item["pair"]
    gt_lbl = "Tương tự" if item["gt"] == 1 else "Khác biệt"
    print(f"{n1 + ' vs ' + n2:<45} | {gt_lbl:<12} | {item['dist_slide']:<15} | {item['dist_std']:<15}")
print(f"{'-'*95}")

# Lưu trữ kết quả so khớp để notebook Phần 2 sử dụng
data_out_dir = "../data/processed"
os.makedirs(data_out_dir, exist_ok=True)
with open(os.path.join(data_out_dir, "pairs_results.json"), "w") as f:
    json.dump({
        "max_len_slide": len(hashes_slide[image_keys[0]]),
        "max_len_std": len(hashes_std[image_keys[0]]),
        "results": pairs_results
    }, f, indent=1)
print("Đã lưu kết quả phân tích sang data/processed/pairs_results.json!")